# NGC4254: extracting conditional KTZ-compatible spiral source fields from the HII SFR map

This notebook takes the geometry and arm-profile model from the first two Python cells of `KTZ_validation.ipynb` and applies it to the `LOGSFR_SURFACE_DENSITY_HII` map in the current NGC4254 gas-bin product. Exactly `m=2,3,4` are compared. Each conditional fit records `m_arms`, negative `pitch_angle`, reference phase `Theta0`, exponential scale `h_R`, modulation `eta`, and harmonic arrays `harmonic_n`, `harmonic_g`, and `harmonic_alpha`.

Negative winding is imposed as the user-approved visual morphology prior. The result is a three-row conditional comparison for a lopsided, disturbed galaxy; no arm-number winner is declared. These models are low-dimensional descriptions rather than evidence for a rigid, stationary density wave, and diffusion, enrichment-age, clustering, and pattern-speed parameters remain outside this fit.


## 1. Model, conventions, and identifiability

The fitted linear source field is `lambda(R, phi) = lambda0_0 exp(-R/h_R) [1 + eta h(Theta)]`, where `Theta = (m/tan(pitch)) ln(R/R_ref) - m phi + Theta0`. The transverse arm profile is a harmonic series, `h(Theta) = sum g_n cos(n Theta + alpha_n)`.

Only the products `eta*g_n` are observable if every harmonic amplitude is free. We therefore fix `g_1 = 1` and `alpha_1 = 0`; the fundamental phase is carried by `Theta0`. This makes `eta` the fundamental modulation amplitude and makes the higher harmonic amplitudes and phases identifiable. Exactly `m=2,3,4` are compared, negative winding is imposed as the user-approved visual morphology prior, and no arm-number winner is declared under the explicit disc-coordinate convention below.


## 2. Imports and visible configuration

This cell contains every path, geometry assumption, optimizer range, and random seed used later. Brown et al. supplies the optical centre, inclination, and directed receding-side position angle. Equation 3 of Huang et al. (2026), https://academic.oup.com/mnras/article/549/3/stag1019/8698768, supplies the finite-thickness surface-density factor that is already applied upstream. Coordinate deprojection still uses `cos(i)`. The FITS WCS handles sky orientation, so the array is not manually flipped north-up/east-left before the tangent-plane rotation.

Exactly `m=2,3,4` are compared, negative winding is imposed as the user-approved visual morphology prior, and no arm-number winner is declared. The implementation contract is `leakage_controlled_negative_winding_ridge`: `conditional_ridge_search` will retain one conditional geometry per arm number in `ridge_geometry_table`, and completed real-data execution will emit `RIDGE_M234_REAL_COMPLETE`. Change these visible values here rather than hiding alternatives inside helper functions.


In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import FK5, SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.optimize import least_squares, minimize_scalar

ROOT = Path.cwd().resolve()
FITS_PATH = ROOT / "v3tk_v7.6.8/NGC4254/NGC4254_gas_bin_maps_further.fits"
BROWN_TABLE = ROOT / "Brown2021Table1.txt"
UPSTREAM_SFR_LOG = ROOT / "sfr_logs/NGC4254.log"
SFR_HDU = "LOGSFR_SURFACE_DENSITY_HII"
BIN_HDU = "BIN_ID"

DISTANCE_MPC = 16.5
Q0 = 0.2
SURFACE_DENSITY_ALREADY_CORRECTED = True
APPLY_BA_CORRECTION_IN_NOTEBOOK = False
R_REF_KPC = 1.0
M_COMPARE = np.array([2, 3, 4], dtype=int)
RIDGE_PITCH_GRID_DEG = np.arange(-45.0, -4.9, 1.0)
RIDGE_GUARD_BINS = 4
RIDGE_NULL_BLOCK_SEEDS = np.array([4254, 5254, 6254, 7254], dtype=int)
RIDGE_NULL_DRAWS_PER_BLOCK = 8
RIDGE_NULL_TOTAL_DRAWS = int(len(RIDGE_NULL_BLOCK_SEEDS) * RIDGE_NULL_DRAWS_PER_BLOCK)
RIDGE_CORE_WIDTH_KPC = 0.25
RIDGE_WIDTH_SENSITIVITY_KPC = np.array([0.18, 0.22, 0.25, 0.30, 0.35])
RIDGE_BROAD_RATIO = 3.0
RIDGE_N_PHASE = 360
RIDGE_N_SECTORS = 12
RIDGE_MIN_HELD_OUT_SECTORS = 10
RIDGE_SHORTLIST_PER_FAMILY = 5
LOGPOLAR_N_U = 120
LOGPOLAR_N_PHI = 360
LOGPOLAR_N_RADIAL_BANDS = 24
LOGPOLAR_SMOOTH_SIGMA = (0.8, 1.2)
LOGPOLAR_AZIMUTH_BROAD_SIGMA_BINS = 30.0
HARMONIC_N = np.array([1, 2, 3], dtype=int)
USE_ALL_VALID_HII_BINS = True
N_BOOTSTRAP = 24
N_BOOTSTRAP_SECTORS = 12
RNG_SEED = 4254
rng = np.random.default_rng(RNG_SEED)


def load_brown_geometry(path, galaxy="NGC 4254"):
    if not path.exists():
        raise FileNotFoundError(path)
    rows = [line.rstrip("\n") for line in path.read_text(encoding="utf-8").splitlines()
            if line.startswith(galaxy)]
    if len(rows) != 1:
        raise RuntimeError(f"Expected one {galaxy} row in {path}; found {len(rows)}")
    fields = rows[0].split("\t")
    if len(fields) < 6:
        raise ValueError(f"Malformed Brown table row: {rows[0]}")
    ra_match = re.fullmatch(r"(\d+)\^h(\d+)\^m(\d+)\.s(\d+)", fields[1])
    dec_match = re.fullmatch(r"([+-]?\d+)deg(\d+)'(\d+)\.\"(\d+)", fields[2])
    if ra_match is None or dec_match is None:
        raise ValueError(f"Unrecognized Brown coordinates: {fields[1:3]}")
    hh, mm, ss, sf = ra_match.groups()
    dd, dm, ds, df = dec_match.groups()
    sign = "-" if dd.startswith("-") else "+"
    dd_abs = dd.lstrip("+-")
    coordinate = SkyCoord(
        f"{hh}h{mm}m{ss}.{sf}s",
        f"{sign}{dd_abs}d{dm}m{ds}.{df}s",
        frame=FK5(equinox="J2000"),
    )
    return {
        "center": coordinate,
        "inclination_deg": float(fields[4]),
        "position_angle_deg": float(fields[5]),
        "source_row": rows[0],
    }


def finite_thickness_axis_ratio(inclination_deg, q0=Q0):
    cosine = np.cos(np.deg2rad(float(inclination_deg)))
    return float(np.sqrt((1.0 - q0**2) * cosine**2 + q0**2))


BROWN_GEOMETRY = load_brown_geometry(BROWN_TABLE)
CENTER = BROWN_GEOMETRY["center"]
INCLINATION_DEG = BROWN_GEOMETRY["inclination_deg"]
POSITION_ANGLE_DEG = BROWN_GEOMETRY["position_angle_deg"]
B_OVER_A = finite_thickness_axis_ratio(INCLINATION_DEG)
EXPECTED_CENTER = SkyCoord("12h18m49.68s", "+14d25m05.52s",
                           frame=FK5(equinox="J2000"))
assert CENTER.separation(EXPECTED_CENTER).to_value(u.arcsec) < 1e-6
assert np.isclose(INCLINATION_DEG, 39.0)
assert np.isclose(POSITION_ANGLE_DEG % 360.0, 243.0)
assert np.isclose(B_OVER_A, 0.787, atol=5e-4)
assert SURFACE_DENSITY_ALREADY_CORRECTED
assert not APPLY_BA_CORRECTION_IN_NOTEBOOK
upstream_text = UPSTREAM_SFR_LOG.read_text(encoding="utf-8")
upstream_matches = re.findall(
    r"Inclination correction ENABLED: applying b/a = ([0-9.]+)",
    upstream_text,
)
if not upstream_matches:
    raise RuntimeError(f"No enabled b/a correction record in {UPSTREAM_SFR_LOG}")
assert np.isclose(float(upstream_matches[-1]), B_OVER_A, atol=5e-4)
print(f"Brown catalog row: {BROWN_GEOMETRY['source_row']}")
print(f"Adopted FK5 J2000 centre: {CENTER.to_string('hmsdms')}")
print(f"Inclination={INCLINATION_DEG:.1f} deg; directed PA={POSITION_ANGLE_DEG:.1f} deg east of north")
print("BROWN_GEOMETRY_PASS")
print(f"Equation-3 finite-thickness factor b/a={B_OVER_A:.4f}; already applied upstream")
print("BA_FACTOR_PASS")
print(f"Confirmed upstream correction record: b/a={float(upstream_matches[-1]):.3f}")
print("UPSTREAM_BA_PASS")

plt.style.use("default")
plt.rcParams.update({"figure.dpi": 115, "font.size": 10})
warnings.filterwarnings("default")
print(f"Project root: {ROOT}")
print(f"Input product: {FITS_PATH}")


## 3. Read the two FITS maps and validate their shared geometry

The SFR map stores logarithmic surface density, while `BIN_ID` records the adaptive gas bin assigned to every image pixel. The loader copies both arrays before the FITS handle closes, verifies identical shapes, and constructs a celestial WCS from the SFR extension. No missing HII pixels are filled, and no science product is changed. The printed counts are live diagnostics rather than fixed sample assumptions.


In [ ]:
def load_maps(path, sfr_hdu=SFR_HDU, bin_hdu=BIN_HDU):
    'Read the log-SFR and bin-ID maps and return an independent celestial WCS.'
    if not path.exists():
        raise FileNotFoundError(path)
    with fits.open(path, memmap=True) as hdul:
        names = {hdu.name for hdu in hdul}
        missing = [name for name in (sfr_hdu, bin_hdu) if name not in names]
        if missing:
            raise KeyError(f"Missing required HDUs: {missing}")
        log_sfr = np.asarray(hdul[sfr_hdu].data, dtype=float).copy()
        bin_id = np.asarray(hdul[bin_hdu].data, dtype=float).copy()
        header = hdul[sfr_hdu].header.copy()
    if log_sfr.shape != bin_id.shape:
        raise ValueError(f"Map shape mismatch: {log_sfr.shape} versus {bin_id.shape}")
    wcs = WCS(header).celestial
    if not wcs.has_celestial:
        raise ValueError("Target HDU does not contain a celestial WCS")
    return log_sfr, bin_id, header, wcs


log_sfr_map, bin_id_map, sfr_header, celestial_wcs = load_maps(FITS_PATH)
wcs_reference = SkyCoord(
    celestial_wcs.wcs.crval[0] * u.deg,
    celestial_wcs.wcs.crval[1] * u.deg,
    frame=FK5(equinox="J2000"),
)
wcs_center_separation_arcsec = float(
    wcs_reference.separation(CENTER).to_value(u.arcsec))
assert wcs_center_separation_arcsec > 1.0
print(f"FITS CRVAL is a WCS projection reference, not the Brown optical centre; "
      f"separation={wcs_center_separation_arcsec:.3f} arcsec")
print("WCS_REFERENCE_NOT_CENTER_PASS")
finite_sfr = np.isfinite(log_sfr_map)
print(f"Map shape: {log_sfr_map.shape}")
print(f"Finite HII SFR pixels: {finite_sfr.sum():,}")
print(f"Finite log SFR range: {np.nanmin(log_sfr_map):.3f} to {np.nanmax(log_sfr_map):.3f}")


## 4. Collapse repeated pixels to one independent gas-bin record

Adaptive gas-bin values are copied to every member pixel, so treating all finite pixels as independent would produce false precision. This function builds one row per valid `BIN_ID`: the mean stored log SFR, pixel centroid, sky centroid, and member-pixel area. Later fits use the member area as an integration weight for independent-bin KTZ fitting and leave-one-sector-out morphology stability.


In [ ]:
def build_bin_catalog(log_sfr, bin_id, wcs):
    'Return one centroid and one SFR measurement per valid adaptive gas bin.'
    valid = np.isfinite(log_sfr) & np.isfinite(bin_id) & (bin_id >= 0)
    yy, xx = np.nonzero(valid)
    ids = bin_id[valid].astype(np.int64)
    values = log_sfr[valid]
    unique_ids, inverse = np.unique(ids, return_inverse=True)
    area_pix = np.bincount(inverse).astype(float)
    x_centroid = np.bincount(inverse, weights=xx) / area_pix
    y_centroid = np.bincount(inverse, weights=yy) / area_pix
    mean_log_sfr = np.bincount(inverse, weights=values) / area_pix
    ra_deg, dec_deg = wcs.pixel_to_world_values(x_centroid, y_centroid)
    table = pd.DataFrame({
        "bin_id": unique_ids,
        "x_pix": x_centroid,
        "y_pix": y_centroid,
        "ra_deg": ra_deg,
        "dec_deg": dec_deg,
        "area_pix": area_pix,
        "log_sfr": mean_log_sfr,
        "sfr_linear": np.power(10.0, mean_log_sfr),
    })
    if len(table) < 100:
        raise ValueError(f"Only {len(table)} valid HII bins; spiral fitting is not supported")
    return table, valid


bin_table, valid_hii_pixels = build_bin_catalog(log_sfr_map, bin_id_map, celestial_wcs)
assert bin_table["bin_id"].is_unique
assert np.all(bin_table["area_pix"] > 0)
assert np.all(np.isfinite(bin_table["sfr_linear"]))
print(f"Independent valid HII gas bins: {len(bin_table):,}")
print(f"Median represented area: {bin_table['area_pix'].median():.0f} image pixels")
display(bin_table.head())


## 5. Inspect the observed map and the adopted sky-plane geometry

Before deprojection, this plot checks that the WCS, optical centre, and position-angle convention are mutually sensible. The white line is the adopted major-axis direction and the cyan line is the perpendicular projected minor axis. A sign or axis-order error here would change the spiral winding and pitch, so this visualization is a required geometry check rather than decorative plotting.


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 7.2))
vmin, vmax = np.nanpercentile(log_sfr_map[finite_sfr], [2, 98])
image = ax.imshow(log_sfr_map, origin="lower", cmap="magma", vmin=vmin, vmax=vmax)
cx, cy = celestial_wcs.world_to_pixel(CENTER)
ax.scatter(cx, cy, marker="+", s=140, linewidth=2.0, color="lime", label="adopted centre")
length_pix = 260.0
pa = np.deg2rad(POSITION_ANGLE_DEG)
# Image x increases approximately east-to-west because CDELT1 is negative; draw axes via sky offsets.
for angle_deg, color, label in [(POSITION_ANGLE_DEG, "white", "major axis"),
                                (POSITION_ANGLE_DEG + 90.0, "cyan", "minor axis")]:
    angle = np.deg2rad(angle_deg)
    dra = np.sin(angle) * 35.0 * u.arcsec
    ddec = np.cos(angle) * 35.0 * u.arcsec
    end1 = CENTER.directional_offset_by(np.arctan2(dra.value, ddec.value) * u.rad,
                                        np.hypot(dra.value, ddec.value) * u.arcsec)
    end2 = CENTER.directional_offset_by((np.arctan2(dra.value, ddec.value) + np.pi) * u.rad,
                                        np.hypot(dra.value, ddec.value) * u.arcsec)
    x1, y1 = celestial_wcs.world_to_pixel(end1)
    x2, y2 = celestial_wcs.world_to_pixel(end2)
    ax.plot([x1, x2], [y1, y2], color=color, lw=1.5, label=label)
ax.set_xlim(0, log_sfr_map.shape[1]-1)
ax.set_ylim(0, log_sfr_map.shape[0]-1)
ax.set_aspect("equal")
ax.set_xlabel("image x (pixel; celestial WCS retained in FITS)")
ax.set_ylabel("image y (pixel)")
ax.set_title("NGC4254 HII log SFR surface density and adopted geometry")
ax.legend(loc="upper right", fontsize=8)
fig.colorbar(image, ax=ax, pad=0.02, label=r"log $\Sigma_{\rm SFR}$")
plt.show()


## 6. Deproject gas-bin centroids into the galaxy plane

Deprojection asks where each sky position would lie if the inclined disc were viewed face-on. Astropy first calculates tangent-plane offsets from the adopted centre: `east` is positive toward increasing right ascension and `north` is positive toward increasing declination. Multiplying the angular offsets by the 16.5 Mpc angular scale converts them to kpc.

The adopted position angle PA is measured east of north. We rotate the sky offsets into the projected galaxy axes using `major = east*sin(PA) + north*cos(PA)` and `projected_minor = -east*cos(PA) + north*sin(PA)`. Inclination shortens only the apparent minor axis, so the face-on coordinate is `minor = projected_minor/cos(inclination)`. Radius and azimuth are then `R = sqrt(major^2 + minor^2)` and `phi = atan2(minor, major)`.

For projecting a fitted skeleton back to the image, the algebra is reversed: `east = major*sin(PA) - projected_minor*cos(PA)` and `north = major*cos(PA) + projected_minor*sin(PA)`, with `projected_minor = minor*cos(inclination)`. The FITS WCS converts those east/north offsets back to image pixels. The centre, inclination, and PA are fixed rather than fitted from the patchy HII map because they are strongly degenerate with spiral phase, pitch, and winding sign.


In [ ]:
ARCSEC_TO_KPC = DISTANCE_MPC * 1.0e3 / 206265.0


def deproject_sky(ra_deg, dec_deg):
    'Convert FK5 J2000 sky coordinates to deprojected major/minor coordinates in kpc.'
    coords = SkyCoord(np.asarray(ra_deg) * u.deg,
                      np.asarray(dec_deg) * u.deg,
                      frame=FK5(equinox="J2000"))
    dlon, dlat = CENTER.spherical_offsets_to(coords)
    east_kpc = dlon.to_value(u.arcsec) * ARCSEC_TO_KPC
    north_kpc = dlat.to_value(u.arcsec) * ARCSEC_TO_KPC
    pa_rad = np.deg2rad(POSITION_ANGLE_DEG)
    inc_rad = np.deg2rad(INCLINATION_DEG)
    major = east_kpc * np.sin(pa_rad) + north_kpc * np.cos(pa_rad)
    minor_projected = -east_kpc * np.cos(pa_rad) + north_kpc * np.sin(pa_rad)
    minor = minor_projected / np.cos(inc_rad)
    radius = np.hypot(major, minor)
    azimuth = np.arctan2(minor, major)
    return major, minor, radius, azimuth


# A direct unit check: the adopted sky centre must map to the disc origin.
origin = deproject_sky([CENTER.ra.deg], [CENTER.dec.deg])
assert np.allclose([origin[0][0], origin[1][0]], 0.0, atol=1.0e-10)

values = deproject_sky(bin_table["ra_deg"], bin_table["dec_deg"])
bin_table[["x_disc_kpc", "y_disc_kpc", "radius_kpc", "azimuth_rad"]] = np.column_stack(values)
# Keep every finite HII bin. The logarithmic spiral is undefined only at exactly R=0;
# no valid bin centroid lies exactly at the adopted centre in this product.
fit_mask = np.isfinite(bin_table["radius_kpc"]) & (bin_table["radius_kpc"] > 0.0)
fit_table = bin_table.loc[fit_mask].reset_index(drop=True)
R_IN_KPC = float(fit_table["radius_kpc"].min())
R_OUT_KPC = float(fit_table["radius_kpc"].max())
assert USE_ALL_VALID_HII_BINS
assert len(fit_table) == len(bin_table)
assert int(fit_table["area_pix"].sum()) == int(valid_hii_pixels.sum())
print(f"1 arcsec = {ARCSEC_TO_KPC:.4f} kpc at {DISTANCE_MPC:.1f} Mpc")
print(f"Fitting radius: {R_IN_KPC:.4f} to {R_OUT_KPC:.4f} kpc; no artificial centre or outer cut")
print(f"Retained bins: {len(fit_table):,} / {len(bin_table):,}")
print(f"Represented valid HII pixels: {int(fit_table['area_pix'].sum()):,} / {int(valid_hii_pixels.sum()):,}")
print("FIT_DOMAIN_ALL_VALID_HII_PASS")

fig, ax = plt.subplots(figsize=(7.5, 7.0))
sc = ax.scatter(fit_table["x_disc_kpc"], fit_table["y_disc_kpc"],
                c=fit_table["log_sfr"], s=np.clip(fit_table["area_pix"]**0.5, 2, 15),
                cmap="magma", vmin=vmin, vmax=vmax, alpha=0.75, linewidth=0)
ax.set_aspect("equal")
ax.set_xlabel("disc major-axis x (kpc)")
ax.set_ylabel("deprojected minor-axis y (kpc)")
ax.set_title("Deprojected independent HII gas bins")
fig.colorbar(sc, ax=ax, label=r"log $\Sigma_{\rm SFR}$")
plt.show()


### 6a. Build all-pixel support for the deprojected ridge morphology

The ridge map needs spatial coverage at image-pixel resolution, so every finite
HII pixel supplies morphology support after WCS deprojection. This is deliberately
separate from the unique-bin catalogue: unique bins remain the independent fitting
records, while repeated member pixels describe where each measurement covers the
disc. The geometry below therefore uses all available HII spaxels, requires only
finite values and the mathematical R > 0 guard, and applies no artificial centre
cut.


In [ ]:
def build_hii_pixel_geometry(log_sfr, valid_mask, wcs):
    yy, xx = np.nonzero(valid_mask)
    ra_deg, dec_deg = wcs.pixel_to_world_values(xx, yy)
    major, minor, radius, azimuth = deproject_sky(ra_deg, dec_deg)
    table = pd.DataFrame({
        "x_pix": xx.astype(float),
        "y_pix": yy.astype(float),
        "x_disc_kpc": major,
        "y_disc_kpc": minor,
        "radius_kpc": radius,
        "azimuth_rad": azimuth,
        "log_sfr": log_sfr[valid_mask],
    })
    keep = np.isfinite(table).all(axis=1) & (table["radius_kpc"] > 0)
    result = table.loc[keep].reset_index(drop=True)
    if len(result) != int(valid_mask.sum()):
        raise RuntimeError(f"Lost {int(valid_mask.sum()) - len(result)} valid HII pixels")
    return result


hii_pixels = build_hii_pixel_geometry(log_sfr_map, valid_hii_pixels, celestial_wcs)
assert len(hii_pixels) == int(valid_hii_pixels.sum())
RIDGE_R_IN_KPC = float(hii_pixels["radius_kpc"].min())
RIDGE_R_OUT_KPC = float(hii_pixels["radius_kpc"].max())
RIDGE_PIVOT_RADIUS_KPC = float(np.exp(
    np.median(np.log(hii_pixels["radius_kpc"].to_numpy(float)))))
print(f"Ridge morphology pixels: {len(hii_pixels):,} / {int(valid_hii_pixels.sum()):,}")
print(f"Ridge pixel radius: {RIDGE_R_IN_KPC:.4f} to {RIDGE_R_OUT_KPC:.4f} kpc; "
      f"phase pivot={RIDGE_PIVOT_RADIUS_KPC:.3f} kpc")
print("RIDGE_PIXEL_DOMAIN_PASS")


## 7. Fit the axisymmetric background and build the log-polar ridge field

The KTZ source field separates into an exponential radial background and a
fractional spiral modulation. Here h_R is the e-folding length of the
axisymmetric SFR source field: increasing radius by h_R lowers that background
by a factor of e. It is not an arm width, pitch angle, optical radius, or
automatically a stellar-disc scale length. The robust fit uses independent bins
and their represented areas, while the morphology field uses every finite HII
pixel.

A live falsification of the earlier coverage rule showed why the mask handling
matters: a global 0.05 * max(coverage) threshold retained only about 14% of
log-polar cells and removed every row below roughly 2.5 kpc, even though those
central pixels entered the histogram. Equal-count quantile rows plus an absolute
local coverage floor retain those pixels, normalize only where local support
exists, and do not create a centre hole. Signed residuals and their negative
flanks are preserved before broad-azimuth structure is removed.


In [ ]:
def fit_radial_exponential(radius, sfr_linear, area_weight):
    'Robustly fit lambda0*exp(-R/h_R) in log space.'
    radius = np.asarray(radius, dtype=float)
    sfr_linear = np.asarray(sfr_linear, dtype=float)
    weights = np.sqrt(np.asarray(area_weight, dtype=float))
    valid = (np.isfinite(radius) & np.isfinite(sfr_linear) &
             np.isfinite(weights) & (sfr_linear > 0) & (weights > 0))
    r = radius[valid]
    y = np.log(sfr_linear[valid])
    w = weights[valid] / np.nanmedian(weights[valid])
    initial = np.array([np.nanmedian(y), np.log(3.0)])

    def residual(theta):
        log_lambda0, log_h = theta
        return w * (y - (log_lambda0 - r / np.exp(log_h)))

    result = least_squares(residual, initial, loss="soft_l1", f_scale=0.3,
                           bounds=([-50.0, np.log(0.2)], [50.0, np.log(30.0)]))
    if not result.success:
        raise RuntimeError(result.message)
    return {"lambda0_0": float(np.exp(result.x[0])),
            "h_R": float(np.exp(result.x[1])),
            "cost": float(2.0 * result.cost), "success": True}


# Synthetic unit check of the radial fitter.
r_test = np.linspace(0.5, 8.0, 200)
radial_test = fit_radial_exponential(r_test, 0.025*np.exp(-r_test/3.4), np.ones_like(r_test))
assert abs(radial_test["h_R"] - 3.4) < 0.02

radial_fit = fit_radial_exponential(fit_table["radius_kpc"],
                                    fit_table["sfr_linear"], fit_table["area_pix"])
radial_fit["gradient_dex_per_kpc"] = float(
    -1.0 / (np.log(10.0) * radial_fit["h_R"]))
print(f"Axisymmetric SFR background h_R={radial_fit['h_R']:.3f} kpc; "
      f"gradient={radial_fit['gradient_dex_per_kpc']:.4f} dex/kpc")



def preprocess_logpolar_raw(
        raw_state, allowed_phi=None, include_local=True,
        smooth_sigma=LOGPOLAR_SMOOTH_SIGMA,
        broad_sigma_phi=LOGPOLAR_AZIMUTH_BROAD_SIGMA_BINS):
    """Validate, mask, and smooth a raw log-polar histogram."""
    required = (
        "raw_weighted_sum", "raw_coverage", "u", "phi",
        "u_edges", "phi_edges",
    )
    missing = [key for key in required if key not in raw_state]
    if missing:
        raise ValueError(f"Raw log-polar state is missing keys: {missing}")

    raw_weighted_sum = np.array(
        raw_state["raw_weighted_sum"], dtype=float, copy=True)
    raw_coverage = np.array(
        raw_state["raw_coverage"], dtype=float, copy=True)
    if raw_weighted_sum.ndim != 2 or raw_coverage.ndim != 2:
        raise ValueError("Raw weighted sum and coverage must be 2-D")
    if raw_weighted_sum.shape != raw_coverage.shape:
        raise ValueError("Raw weighted sum and coverage shapes differ")
    if not np.isfinite(raw_weighted_sum).all():
        raise ValueError("Raw weighted sum must be finite")
    if not np.isfinite(raw_coverage).all() or np.any(raw_coverage < 0.0):
        raise ValueError("Raw coverage must be finite and non-negative")

    u = np.array(raw_state["u"], dtype=float, copy=True)
    phi = np.array(raw_state["phi"], dtype=float, copy=True)
    u_edges = np.array(raw_state["u_edges"], dtype=float, copy=True)
    phi_edges = np.array(raw_state["phi_edges"], dtype=float, copy=True)
    n_u, n_phi = raw_weighted_sum.shape
    if u.ndim != 1 or u.size != n_u:
        raise ValueError("u must be a 1-D coordinate matching the radial axis")
    if phi.ndim != 1 or phi.size != n_phi:
        raise ValueError("phi must be a 1-D coordinate matching the azimuth axis")
    if u_edges.ndim != 1 or u_edges.size != n_u + 1:
        raise ValueError("u_edges must have one more entry than the radial axis")
    if phi_edges.ndim != 1 or phi_edges.size != n_phi + 1:
        raise ValueError(
            "phi_edges must have one more entry than the azimuth axis")
    for label, values in (
            ("u", u), ("phi", phi),
            ("u_edges", u_edges), ("phi_edges", phi_edges)):
        if not np.isfinite(values).all():
            raise ValueError(f"{label} must be finite")
    if np.any(np.diff(u_edges) <= 0.0):
        raise ValueError("u_edges must be strictly increasing")
    if np.any(np.diff(phi_edges) <= 0.0):
        raise ValueError("phi_edges must be strictly increasing")

    allowed = None
    if allowed_phi is not None:
        allowed = np.array(allowed_phi, dtype=bool, copy=True)
        if allowed.ndim != 1 or allowed.size != n_phi:
            raise ValueError("allowed_phi must match the log-polar azimuth axis")
        raw_weighted_sum[:, ~allowed] = 0.0
        raw_coverage[:, ~allowed] = 0.0

    numerator = gaussian_filter(
        raw_weighted_sum, smooth_sigma, mode=("nearest", "wrap"))
    denominator = gaussian_filter(
        raw_coverage, smooth_sigma, mode=("nearest", "wrap"))
    valid = (
        (denominator > 0.05)
        & np.isfinite(numerator)
        & np.isfinite(denominator)
    )
    radial_residual = np.divide(
        numerator, denominator, out=np.full_like(numerator, np.nan),
        where=valid)

    broad_azimuthal = np.full_like(radial_residual, np.nan)
    local_ridge = np.full_like(radial_residual, np.nan)
    if include_local:
        broad_numerator = gaussian_filter1d(
            np.where(valid, radial_residual * denominator, 0.0),
            broad_sigma_phi, axis=1, mode="wrap")
        broad_denominator = gaussian_filter1d(
            np.where(valid, denominator, 0.0),
            broad_sigma_phi, axis=1, mode="wrap")
        broad_azimuthal = np.divide(
            broad_numerator, broad_denominator,
            out=np.full_like(broad_numerator, np.nan),
            where=broad_denominator > 0.0)
        local_ridge[valid] = (
            radial_residual[valid] - broad_azimuthal[valid])

    if allowed is not None:
        disallowed = ~allowed
        valid[:, disallowed] = False
        denominator[:, disallowed] = 0.0
        radial_residual[:, disallowed] = np.nan
        local_ridge[:, disallowed] = np.nan
        broad_azimuthal[:, disallowed] = np.nan

    return {
        "radial_residual": radial_residual,
        "local_ridge": local_ridge,
        "coverage": denominator,
        "valid": valid,
        "u": u,
        "phi": phi,
        "u_edges": u_edges,
        "phi_edges": phi_edges,
        "broad_azimuthal": broad_azimuthal,
    }


def build_log_polar_contrast(
        pixel_table, radial_parameters,
        n_u=LOGPOLAR_N_U, n_phi=LOGPOLAR_N_PHI):
    """Build unsmoothed log-polar sums, then derive display fields."""
    n_u = int(n_u)
    n_phi = int(n_phi)
    if n_u < 1 or n_phi < 1:
        raise ValueError("Log-polar dimensions must be positive")

    radius = pixel_table["radius_kpc"].to_numpy(float)
    azimuth = pixel_table["azimuth_rad"].to_numpy(float)
    observed_log = pixel_table["log_sfr"].to_numpy(float)
    if not (radius.ndim == azimuth.ndim == observed_log.ndim == 1):
        raise ValueError("Log-polar pixel columns must be one-dimensional")
    if not (radius.size == azimuth.size == observed_log.size) or radius.size == 0:
        raise ValueError("Log-polar pixel columns must have equal nonzero length")
    if not np.isfinite(radius).all() or np.any(radius <= 0.0):
        raise ValueError("Log-polar radii must be finite and positive")
    if not np.isfinite(azimuth).all():
        raise ValueError("Log-polar azimuths must be finite")
    if not np.isfinite(observed_log).all():
        raise ValueError("Log-polar SFR values must be finite")

    lambda0_0 = float(radial_parameters["lambda0_0"])
    h_r = float(radial_parameters["h_R"])
    if not np.isfinite(lambda0_0) or lambda0_0 <= 0.0:
        raise ValueError("lambda0_0 must be finite and positive")
    if not np.isfinite(h_r) or h_r <= 0.0:
        raise ValueError("h_R must be finite and positive")

    background_log = (
        np.log10(lambda0_0) - radius / (np.log(10.0) * h_r))
    signed_residual = observed_log - background_log
    u = np.log(radius / R_REF_KPC)
    if not np.isfinite(u).all() or not np.isfinite(signed_residual).all():
        raise ValueError("Derived log-polar coordinates must be finite")

    # Validate repeated quantiles before nextafter expands the endpoints.
    quantile_edges = np.quantile(u, np.linspace(0.0, 1.0, n_u + 1))
    if np.any(np.diff(quantile_edges) <= 0.0):
        raise ValueError(
            "Quantile log-radius edges are not strictly increasing")
    u_edges = np.array(quantile_edges, copy=True)
    u_edges[0] = np.nextafter(u_edges[0], -np.inf)
    u_edges[-1] = np.nextafter(u_edges[-1], np.inf)
    phi_edges = np.linspace(-np.pi, np.pi, n_phi + 1)

    raw_weighted_sum, _, _ = np.histogram2d(
        u, azimuth, bins=(u_edges, phi_edges), weights=signed_residual)
    raw_coverage, _, _ = np.histogram2d(
        u, azimuth, bins=(u_edges, phi_edges))
    row_index = np.clip(
        np.searchsorted(u_edges, u, side="right") - 1, 0, n_u - 1)
    row_count = np.bincount(row_index, minlength=n_u)
    if np.any(row_count <= 0):
        raise ValueError("Every log-radius row must contain at least one pixel")
    u_centres = (
        np.bincount(row_index, weights=u, minlength=n_u) / row_count)
    if not np.isfinite(u_centres).all():
        raise ValueError("Log-radius row centres must be finite")

    raw_state = {
        "raw_weighted_sum": raw_weighted_sum,
        "raw_coverage": raw_coverage,
        "u": u_centres,
        "phi": 0.5 * (phi_edges[:-1] + phi_edges[1:]),
        "u_edges": u_edges,
        "phi_edges": phi_edges,
    }
    return raw_state, preprocess_logpolar_raw(raw_state, include_local=True)


logpolar_raw, logpolar = build_log_polar_contrast(hii_pixels, radial_fit)
assert np.isfinite(logpolar["radial_residual"][logpolar["valid"]]).all()
assert np.isfinite(logpolar["local_ridge"][logpolar["valid"]]).all()
assert int(logpolar_raw["raw_coverage"].sum()) == len(hii_pixels)
assert len(hii_pixels) == int(valid_hii_pixels.sum())

fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.2), constrained_layout=True,
                         sharex=True, sharey=True)
for ax, field, title in zip(
        axes, ["radial_residual", "local_ridge"],
        ["signed residual after radial background",
         "local ridge field after broad-azimuth removal"]):
    image = ax.pcolormesh(
        logpolar["phi_edges"], logpolar["u_edges"], logpolar[field],
        shading="auto", cmap="coolwarm")
    ax.set(xlabel=r"disc azimuth $\phi$ (rad)",
           ylabel=r"$\ln(R/R_{\rm ref})$", title=title)
    fig.colorbar(image, ax=ax, label="residual (dex)")
plt.show()


radial_model = radial_fit["lambda0_0"] * np.exp(-fit_table["radius_kpc"].to_numpy()/radial_fit["h_R"])
fit_table["radial_model"] = radial_model
fit_table["q"] = fit_table["sfr_linear"].to_numpy()/radial_model - 1.0
print(radial_fit)

radial_edges = np.linspace(R_IN_KPC, R_OUT_KPC, 28)
radial_index = np.digitize(fit_table["radius_kpc"], radial_edges) - 1
summary_rows = []
for idx in range(len(radial_edges)-1):
    use = radial_index == idx
    if use.sum() >= 10:
        summary_rows.append((np.median(fit_table.loc[use, "radius_kpc"]),
                             np.median(fit_table.loc[use, "log_sfr"]), use.sum()))
radial_summary = pd.DataFrame(summary_rows, columns=["radius_kpc", "median_log_sfr", "n"])

fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.scatter(fit_table["radius_kpc"], fit_table["log_sfr"], s=2, alpha=0.06, color="tab:blue")
ax.plot(radial_summary["radius_kpc"], radial_summary["median_log_sfr"], "o", color="black", label="radial medians")
r_line = np.linspace(R_IN_KPC, R_OUT_KPC, 400)
ax.plot(r_line, np.log10(radial_fit["lambda0_0"]*np.exp(-r_line/radial_fit["h_R"])),
        color="tab:red", lw=2.2, label=fr"fit: $h_R={radial_fit['h_R']:.2f}$ kpc")
ax.set(xlabel="deprojected radius (kpc)", ylabel=r"log $\Sigma_{\rm SFR}$",
       title="Axisymmetric exponential background")
ax.legend()
plt.show()


## 8. Define the KTZ spiral phase and select arm number, winding, and phase from SFR ridges

The signed logarithmic-spiral phase maps a coherent ridge into a nearly constant log-polar phase. Its physical transverse distance obeys

`d_perp approximately R * abs(sin(pitch)) * abs(Theta) / m`

so a constant physical corridor must use the radius-dependent width

`sigma_Theta(R) = m * width_kpc / (R * abs(sin(pitch))).`

The rejected implementation used one phase width fixed at `R_ref`: at `R=5` and `10` kpc a nominal 0.35-kpc corridor broadened to approximately 1.75 and 3.5 kpc, respectively, rewarding lopsidedness rather than a narrow arm ridge. The replacement measures a narrow phase-smoothed mean minus a broad local-flank mean in approximately equal-count radial bands. Dividing weighted signal by coverage makes each branch a mean, not a sum, so increasing `m_arms` does not win merely by contributing more repeated branches.

Every `m/sign/pitch` candidate is initially screened on the full map. Exactly five pitches per `m` and winding sign enter screened azimuth-sector validation: each held-out sector is scored at a phase learned from the remaining sectors. This is screened sector validation, not fully nested cross-validation and not an unbiased posterior. The validated ridge score freezes `m_arms`, signed `pitch_angle`, and `Theta0`; no later optimizer is allowed to refine or relabel that geometry.

In [ ]:

def sector_columns(phi, n_sectors=RIDGE_N_SECTORS):
    """Map log-polar azimuth columns onto clipped sector indices."""
    phi = np.asarray(phi, dtype=float)
    sector = np.floor(
        (phi + np.pi) / (2.0 * np.pi) * int(n_sectors)).astype(int)
    return np.clip(sector, 0, int(n_sectors) - 1)


def circular_dilate(mask, guard_bins):
    """Dilate a one-dimensional circular mask by guard_bins columns."""
    mask = np.asarray(mask, dtype=bool)
    guard_bins = int(guard_bins)
    if mask.ndim != 1 or guard_bins < 0:
        raise ValueError("Circular dilation needs a 1D mask and non-negative guard")
    shifted = [np.roll(mask, shift)
               for shift in range(-guard_bins, guard_bins + 1)]
    return np.logical_or.reduce(shifted)


def circular_erode(mask, guard_bins):
    """Erode a one-dimensional circular mask by guard_bins columns."""
    mask = np.asarray(mask, dtype=bool)
    guard_bins = int(guard_bins)
    if mask.ndim != 1 or guard_bins < 0:
        raise ValueError("Circular erosion needs a 1D mask and non-negative guard")
    shifted = [np.roll(mask, shift)
               for shift in range(-guard_bins, guard_bins + 1)]
    return np.logical_and.reduce(shifted)


def logpolar_fold_masks(
        phi, held_sector, n_sectors=RIDGE_N_SECTORS,
        guard_bins=RIDGE_GUARD_BINS):
    """Return held, guarded-training, and eroded-test azimuth masks."""
    held_sector = int(held_sector)
    n_sectors = int(n_sectors)
    if held_sector < 0 or held_sector >= n_sectors:
        raise ValueError("held_sector lies outside the requested sector range")
    held_columns = sector_columns(phi, n_sectors) == held_sector
    train_allowed = ~circular_dilate(held_columns, guard_bins)
    test_allowed = circular_erode(held_columns, guard_bins)
    if np.any(train_allowed & test_allowed):
        raise RuntimeError("Training and test azimuth masks overlap")
    return {
        "held_columns": held_columns,
        "train_allowed": train_allowed,
        "test_allowed": test_allowed,
    }


def build_sector_fold_maps(
        raw_state, n_sectors=RIDGE_N_SECTORS,
        guard_bins=RIDGE_GUARD_BINS):
    """Split raw azimuth columns before independently smoothing each fold."""
    folds = []
    for held_sector in range(int(n_sectors)):
        masks = logpolar_fold_masks(
            raw_state["phi"], held_sector, n_sectors, guard_bins)
        train = preprocess_logpolar_raw(
            raw_state, allowed_phi=masks["train_allowed"],
            include_local=False)
        test = preprocess_logpolar_raw(
            raw_state, allowed_phi=masks["test_allowed"],
            include_local=False)
        folds.append({
            "held_sector": held_sector,
            "held_columns": masks["held_columns"],
            "train_allowed": masks["train_allowed"],
            "test_allowed": masks["test_allowed"],
            "train": train,
            "test": test,
        })
    return folds


def run_adversarial_leakage_check(
        n_u=12, n_phi=48, n_sectors=4,
        held_sector=1, guard_bins=2):
    """Test held-signal exclusion and a known nonzero training control."""
    n_u = int(n_u)
    n_phi = int(n_phi)
    u_edges = np.linspace(-1.0, 1.0, n_u + 1)
    phi_edges = np.linspace(-np.pi, np.pi, n_phi + 1)
    u_centres = 0.5 * (u_edges[:-1] + u_edges[1:])
    phi_centres = 0.5 * (phi_edges[:-1] + phi_edges[1:])
    masks = logpolar_fold_masks(
        phi_centres, held_sector, n_sectors, guard_bins)
    held_columns = masks["held_columns"]
    train_allowed = masks["train_allowed"]
    raw_coverage = np.full((n_u, n_phi), 20.0)

    def make_raw_state(weighted_sum):
        return {
            "raw_weighted_sum": np.array(weighted_sum, copy=True),
            "raw_coverage": np.array(raw_coverage, copy=True),
            "u": u_centres,
            "phi": phi_centres,
            "u_edges": u_edges,
            "phi_edges": phi_edges,
        }

    # First probe: all signal is confined to the held sector. It must leave
    # no residual in the guarded training field.
    held_only_weighted = np.zeros_like(raw_coverage)
    held_only_weighted[:, held_columns] = raw_coverage[:, held_columns]
    held_only_state = make_raw_state(held_only_weighted)
    fold = build_sector_fold_maps(
        held_only_state, n_sectors=n_sectors,
        guard_bins=guard_bins)[int(held_sector)]
    train = fold["train"]
    raw_training_signal_l1 = float(np.sum(np.abs(
        held_only_weighted[:, train_allowed])))
    training_values = train["radial_residual"][train["valid"]]
    max_abs_training_residual = (
        float(np.max(np.abs(training_values)))
        if training_values.size else 0.0)

    # Independent reference: physically excise every disallowed raw column,
    # then smooth without passing an allowed mask.
    reference_state = {
        key: np.array(value, copy=True)
        for key, value in held_only_state.items()
    }
    reference_state["raw_weighted_sum"][:, ~train_allowed] = 0.0
    reference_state["raw_coverage"][:, ~train_allowed] = 0.0
    reference = preprocess_logpolar_raw(
        reference_state, allowed_phi=None, include_local=False)
    training_support = np.broadcast_to(
        train_allowed[None, :], train["valid"].shape)
    common_valid = training_support & train["valid"] & reference["valid"]
    train_valid_count = int(np.count_nonzero(common_valid))
    max_abs_residual_difference = (
        float(np.max(np.abs(
            train["radial_residual"][common_valid]
            - reference["radial_residual"][common_valid])))
        if train_valid_count else np.inf)
    max_abs_coverage_difference = float(np.max(np.abs(
        train["coverage"][:, train_allowed]
        - reference["coverage"][:, train_allowed])))

    # Second probe: a constant, independently known training residual ensures
    # that an implementation cannot pass by discarding all weighted data.
    expected_training_residual = 0.25
    known_weighted = np.zeros_like(raw_coverage)
    known_weighted[:, train_allowed] = (
        expected_training_residual * raw_coverage[:, train_allowed])
    known_weighted[:, held_columns] = raw_coverage[:, held_columns]
    known_state = make_raw_state(known_weighted)
    known_train = build_sector_fold_maps(
        known_state, n_sectors=n_sectors,
        guard_bins=guard_bins)[int(held_sector)]["train"]
    known_values = known_train["radial_residual"][known_train["valid"]]
    known_train_valid_count = int(known_values.size)
    mean_known_training_residual = (
        float(np.mean(known_values)) if known_values.size else np.nan)
    max_abs_known_training_error = (
        float(np.max(np.abs(
            known_values - expected_training_residual)))
        if known_values.size else np.inf)

    return {
        "raw_training_signal_l1": raw_training_signal_l1,
        "max_abs_training_residual": max_abs_training_residual,
        "train_valid_count": train_valid_count,
        "max_abs_residual_difference": max_abs_residual_difference,
        "max_abs_coverage_difference": max_abs_coverage_difference,
        "known_train_valid_count": known_train_valid_count,
        "expected_training_residual": expected_training_residual,
        "mean_known_training_residual": mean_known_training_residual,
        "max_abs_known_training_error": max_abs_known_training_error,
    }


adversarial_leakage = run_adversarial_leakage_check()
assert adversarial_leakage["raw_training_signal_l1"] == 0.0
assert adversarial_leakage["max_abs_training_residual"] <= 1.0e-14
assert adversarial_leakage["train_valid_count"] > 0
assert adversarial_leakage["max_abs_residual_difference"] <= 1.0e-14
assert adversarial_leakage["max_abs_coverage_difference"] <= 1.0e-14
assert adversarial_leakage["known_train_valid_count"] > 0
assert abs(adversarial_leakage["mean_known_training_residual"]
           - adversarial_leakage["expected_training_residual"]) <= 1.0e-14
assert adversarial_leakage["max_abs_known_training_error"] <= 1.0e-14
print("RIDGE_LEAKAGE_GUARD_PASS")


def wrap_angle(angle):
    return np.angle(np.exp(1j*np.asarray(angle)))


def spiral_phase(radius, azimuth, m_arms, pitch_angle, theta0=0.0):
    pitch = np.deg2rad(pitch_angle)
    return ((m_arms/np.tan(pitch))*np.log(np.asarray(radius)/R_REF_KPC)
            - m_arms*np.asarray(azimuth) + theta0)


def arm_profile(theta, harmonic_g, harmonic_alpha):
    theta = np.asarray(theta, dtype=float)
    value = np.zeros_like(theta)
    for n, g, alpha in zip(HARMONIC_N, harmonic_g, harmonic_alpha):
        value += g*np.cos(n*theta + alpha)
    return value


def phase_sector_row_histograms(
        logpolar_map, m_arms, pitch_angle, field="local_ridge",
        n_phase=RIDGE_N_PHASE, n_sectors=RIDGE_N_SECTORS):
    uu, pp = np.meshgrid(logpolar_map["u"], logpolar_map["phi"], indexing="ij")
    valid = logpolar_map["valid"] & np.isfinite(logpolar_map[field])
    n_u = len(logpolar_map["u"])
    u = uu[valid]
    phi = pp[valid]
    row = np.broadcast_to(np.arange(n_u)[:, None], valid.shape)[valid]
    ridge_value = logpolar_map[field][valid]
    coverage = logpolar_map["coverage"][valid]
    base_phase = np.mod((m_arms / np.tan(np.deg2rad(pitch_angle))) * u
                        - m_arms * phi, 2.0 * np.pi)
    phase_index = np.floor(base_phase / (2.0 * np.pi) * n_phase).astype(int) % n_phase
    sector_index = np.floor((phi + np.pi) / (2.0 * np.pi) * n_sectors).astype(int)
    sector_index = np.clip(sector_index, 0, n_sectors - 1)
    joint_index = (sector_index * n_u + row) * n_phase + phase_index
    size = n_sectors * n_u * n_phase
    weighted = np.bincount(joint_index, weights=coverage * ridge_value,
                           minlength=size).reshape(n_sectors, n_u, n_phase)
    support = np.bincount(joint_index, weights=coverage,
                          minlength=size).reshape(n_sectors, n_u, n_phase)
    return weighted, support


def variable_width_ridge_response(
        weighted_hist, support_hist, u_centres, m_arms, pitch_angle,
        core_width_kpc=RIDGE_CORE_WIDTH_KPC,
        broad_ratio=RIDGE_BROAD_RATIO,
        n_radial_bands=LOGPOLAR_N_RADIAL_BANDS):
    """Local centre-minus-flanks response at a constant physical width."""
    n_u, n_phase = weighted_hist.shape
    if n_u % n_radial_bands:
        raise ValueError("LOGPOLAR_N_U must be divisible by radial-band count")
    rows_per_band = n_u // n_radial_bands
    weighted_band = weighted_hist.reshape(
        n_radial_bands, rows_per_band, n_phase).sum(axis=1)
    support_band = support_hist.reshape(
        n_radial_bands, rows_per_band, n_phase).sum(axis=1)
    u_band = np.asarray(u_centres).reshape(
        n_radial_bands, rows_per_band).mean(axis=1)
    radius_band = R_REF_KPC * np.exp(u_band)
    sine_pitch = abs(np.sin(np.deg2rad(pitch_angle)))
    sigma_phase = m_arms * core_width_kpc / (radius_band * sine_pitch)
    sigma_bins = np.clip(
        sigma_phase / (2.0 * np.pi) * n_phase, 0.65, n_phase / 10.0)

    narrow_mean = np.full_like(weighted_band, np.nan, dtype=float)
    broad_mean = np.full_like(weighted_band, np.nan, dtype=float)
    narrow_support = np.zeros_like(support_band, dtype=float)
    for radial_index, sigma_bin in enumerate(sigma_bins):
        narrow_weighted = gaussian_filter1d(
            weighted_band[radial_index], sigma_bin, mode="wrap")
        narrow_denominator = gaussian_filter1d(
            support_band[radial_index], sigma_bin, mode="wrap")
        broad_weighted = gaussian_filter1d(
            weighted_band[radial_index], broad_ratio * sigma_bin, mode="wrap")
        broad_denominator = gaussian_filter1d(
            support_band[radial_index], broad_ratio * sigma_bin, mode="wrap")
        narrow_mean[radial_index] = np.divide(
            narrow_weighted, narrow_denominator,
            out=np.full(n_phase, np.nan), where=narrow_denominator > 0.05)
        broad_mean[radial_index] = np.divide(
            broad_weighted, broad_denominator,
            out=np.full(n_phase, np.nan), where=broad_denominator > 0.05)
        narrow_support[radial_index] = narrow_denominator
    return narrow_mean - broad_mean, narrow_mean, broad_mean, narrow_support


def aggregate_radial_response(response, support, min_radial_fraction=0.35):
    """Give each approximately equal-count radial band one vote."""
    good = np.isfinite(response) & (support > 0.05)
    count = good.sum(axis=0)
    total = np.where(good, response, 0.0).sum(axis=0)
    mean_response = np.divide(
        total, count, out=np.full(response.shape[1], np.nan), where=count > 0)
    positive_count = (good & (response > 0)).sum(axis=0)
    positive_fraction = np.divide(
        positive_count, count,
        out=np.full(response.shape[1], np.nan), where=count > 0)
    median_response = np.array([
        np.median(response[good[:, index], index]) if count[index] else np.nan
        for index in range(response.shape[1])
    ])
    radial_fraction = count / response.shape[0]
    combined = mean_response * positive_fraction + 0.5 * median_response
    combined[radial_fraction < min_radial_fraction] = np.nan
    return {
        "score": combined,
        "mean": mean_response,
        "median": median_response,
        "positive_fraction": positive_fraction,
        "radial_fraction": radial_fraction,
    }


def held_out_ridge_search(logpolar_map, m_candidates=M_CANDIDATES,
                          pitch_grid=RIDGE_PITCH_GRID_DEG,
                          core_width_kpc=RIDGE_CORE_WIDTH_KPC,
                          allow_no_acceptance=False):
    records = []
    histogram_cache = {}
    branch_normalization = "mean_not_sum"
    phase_values = np.linspace(0.0, 2.0 * np.pi, RIDGE_N_PHASE, endpoint=False)
    for m_arms in m_candidates:
        for pitch_angle in pitch_grid:
            weighted, support = phase_sector_row_histograms(
                logpolar_map, int(m_arms), float(pitch_angle))
            histogram_cache[(int(m_arms), float(pitch_angle))] = (weighted, support)
            total_weighted = weighted.sum(axis=0)
            total_support = support.sum(axis=0)
            response, narrow, broad, response_support = variable_width_ridge_response(
                total_weighted, total_support, logpolar_map["u"],
                int(m_arms), float(pitch_angle), core_width_kpc)
            full = aggregate_radial_response(response, response_support)
            if not np.isfinite(full["score"]).any():
                continue
            full_index = int(np.nanargmax(full["score"]))
            records.append({
                "m_arms": int(m_arms),
                "pitch_angle": float(pitch_angle),
                "winding_sign": int(np.sign(pitch_angle)),
                "Theta0": float((-phase_values[full_index]) % (2.0*np.pi)),
                "core_width_kpc": float(core_width_kpc),
                "full_ridge_score": float(full["score"][full_index]),
                "narrow_mean": float(np.nanmean(narrow[:, full_index])),
                "broad_flank_mean": float(np.nanmean(broad[:, full_index])),
                "positive_radial_fraction": float(full["positive_fraction"][full_index]),
                "radial_coverage": float(full["radial_fraction"][full_index]),
                "branch_normalization": branch_normalization,
                "held_out_score": np.nan,
                "held_out_score_std": np.nan,
                "phase_stability": np.nan,
                "valid_held_out": 0,
                "validated_score": np.nan,
            })
    table = pd.DataFrame(records)
    if table.empty:
        raise RuntimeError("No ridge candidate had finite full-map support")
    shortlist = (table.sort_values("full_ridge_score", ascending=False)
                 .groupby(["m_arms", "winding_sign"], group_keys=False)
                 .head(RIDGE_SHORTLIST_PER_FAMILY).index)
    for record_index in shortlist:
        m_arms = int(table.at[record_index, "m_arms"])
        pitch_angle = float(table.at[record_index, "pitch_angle"])
        weighted, support = histogram_cache[(m_arms, pitch_angle)]
        total_weighted = weighted.sum(axis=0)
        total_support = support.sum(axis=0)
        held_out_scores = []
        held_out_phases = []
        for sector in range(RIDGE_N_SECTORS):
            train_response, _, _, train_support = variable_width_ridge_response(
                total_weighted - weighted[sector], total_support - support[sector],
                logpolar_map["u"], m_arms, pitch_angle, core_width_kpc)
            train = aggregate_radial_response(train_response, train_support)
            if not np.isfinite(train["score"]).any():
                continue
            train_index = int(np.nanargmax(train["score"]))
            test_response, _, _, test_support = variable_width_ridge_response(
                weighted[sector], support[sector], logpolar_map["u"],
                m_arms, pitch_angle, core_width_kpc)
            test = aggregate_radial_response(
                test_response, test_support, min_radial_fraction=0.08)
            if np.isfinite(test["score"][train_index]):
                held_out_scores.append(float(test["score"][train_index]))
                held_out_phases.append(float((-phase_values[train_index]) % (2*np.pi)))
        valid_held_out = len(held_out_scores)
        if valid_held_out:
            phase_stability = float(np.abs(np.mean(
                np.exp(1j * np.asarray(held_out_phases)))))
            held_out_score = float(np.median(held_out_scores))
            table.at[record_index, "held_out_score"] = held_out_score
            table.at[record_index, "held_out_score_std"] = float(np.std(held_out_scores))
            table.at[record_index, "phase_stability"] = phase_stability
            table.at[record_index, "valid_held_out"] = valid_held_out
            table.at[record_index, "validated_score"] = (
                max(held_out_score, 0.0) * (valid_held_out/RIDGE_N_SECTORS)
                * phase_stability)
    accepted = table.loc[
        (table["valid_held_out"] >= RIDGE_MIN_HELD_OUT_SECTORS)
        & (table["validated_score"] > 0)].copy()
    if accepted.empty and not allow_no_acceptance:
        raise RuntimeError("No ridge candidate passed sector-support acceptance")
    ranked = table.sort_values(["validated_score", "full_ridge_score"],
                               ascending=False, na_position="last").reset_index(drop=True)
    if accepted.empty:
        return None, ranked
    accepted = accepted.sort_values(["validated_score", "full_ridge_score"],
                                    ascending=False)
    return accepted.iloc[0].to_dict(), ranked

## 9. Reserve the source-profile fit for the frozen ridge geometry

The Fourier-seeded harmonic regression and full-field geometry optimizer have been deliberately removed because they could change arm number, winding, pitch, or phase after ridge validation. Task 4 will introduce a source-profile fit that conditions on the frozen `ridge_geometry` selected below. This executable placeholder performs no fit and preserves the notebook cell structure until that fixed-geometry implementation is added.

In [ ]:
print("FIXED_GEOMETRY_PROFILE_PENDING_TASK_4")

## 10. Synthetic ridge-recovery for arm number, winding, and phase

These masked synthetic SFR fields combine a known exponential radial decline with narrow logarithmic ridges. Six cases cover every candidate arm number from one through six and both winding signs. The exact production ridge selector must recover arm number, signed pitch within two degrees, and phase within five degrees. A separate scaling check verifies that multiplying both the weighted histogram and its support leaves the mean-normalized ridge response unchanged.

In [ ]:
def synthetic_sfr_pixels(m_arms, pitch_angle, theta0, seed,
                         n_pixels=80000, mask_fraction=0.25,
                         arm_amplitude_dex=0.48,
                         ridge_sigma_kpc=0.22, radial_h_kpc=3.5):
    """Masked SFR pixels with a known radial decline and logarithmic ridges."""
    local_rng = np.random.default_rng(seed)
    u = local_rng.uniform(np.log(0.35), np.log(11.0), n_pixels)
    azimuth = local_rng.uniform(-np.pi, np.pi, n_pixels)
    radius = R_REF_KPC * np.exp(u)
    theta = ((m_arms / np.tan(np.deg2rad(pitch_angle))) * u
             - m_arms * azimuth + theta0)
    wrapped = wrap_angle(theta)
    perpendicular_kpc = (np.abs(wrapped) * radius
                         * abs(np.sin(np.deg2rad(pitch_angle))) / m_arms)
    ridge_dex = arm_amplitude_dex * np.exp(
        -0.5 * (perpendicular_kpc / ridge_sigma_kpc)**2)
    lambda0_0 = 0.05
    background_log = (np.log10(lambda0_0)
                      - radius / (np.log(10.0) * radial_h_kpc))
    observed_log = background_log + ridge_dex
    observed_log += local_rng.normal(0.0, 0.035, size=n_pixels)
    keep = local_rng.random(n_pixels) >= mask_fraction
    keep &= ~((azimuth > 0.35) & (azimuth < 0.70) & (radius > 5.5))
    keep &= ~((azimuth > -2.20) & (azimuth < -1.95) & (radius < 2.2))
    pixels = pd.DataFrame({"radius_kpc": radius[keep],
                           "azimuth_rad": azimuth[keep],
                           "log_sfr": observed_log[keep]})
    return pixels, {"lambda0_0": lambda0_0, "h_R": radial_h_kpc}


def synthetic_logpolar(m_arms, pitch_angle, theta0, seed, **kwargs):
    pixels, radial_parameters = synthetic_sfr_pixels(
        m_arms, pitch_angle, theta0, seed, **kwargs)
    _, processed = build_log_polar_contrast(pixels, radial_parameters)
    return processed


synthetic_cases = [(1, -18.0), (2, 21.0), (3, -24.0),
                   (4, 27.0), (5, -30.0), (6, 33.0)]
theta0_true = 0.7
synthetic_recoveries = []
for case_index, (m_true, pitch_true) in enumerate(synthetic_cases):
    synthetic = synthetic_logpolar(m_true, pitch_true, theta0_true,
                                   RNG_SEED + case_index)
    recovered, _ = held_out_ridge_search(
        synthetic, m_candidates=M_COMPARE,
        pitch_grid=RIDGE_PITCH_GRID_DEG)
    synthetic_recoveries.append(recovered)
    assert recovered["m_arms"] == m_true, (m_true, recovered)
    assert np.sign(recovered["pitch_angle"]) == np.sign(pitch_true), recovered
    assert abs(recovered["pitch_angle"] - pitch_true) <= 2.0, recovered
    phase_error = abs(wrap_angle(recovered["Theta0"] - theta0_true))
    assert phase_error <= np.deg2rad(5.0), (m_true, phase_error, recovered)
print("RIDGE_SYNTHETIC_SIGN_PASS")
print("RIDGE_SYNTHETIC_M_PASS")
print("RIDGE_SYNTHETIC_PHASE_PASS")
assert synthetic_recoveries[0]["m_arms"] == 1
print("RIDGE_M_NORMALIZATION_PASS")

test_weighted, test_support = phase_sector_row_histograms(synthetic, 3, -24.0)
response_1, _, _, _ = variable_width_ridge_response(
    test_weighted.sum(axis=0), test_support.sum(axis=0), synthetic["u"], 3, -24.0)
response_6, _, _, _ = variable_width_ridge_response(
    6.0 * test_weighted.sum(axis=0), 6.0 * test_support.sum(axis=0),
    synthetic["u"], 3, -24.0)
assert np.allclose(response_1, response_6, equal_nan=True, atol=1e-12)
print("RIDGE_BRANCH_NORMALIZATION_PASS")

## 11. Select the real NGC4254 ridge geometry, then fit its source profile

All valid HII pixels select the geometry from the signed local-ridge field and screened azimuth-sector validation. Independent gas bins will later fit the full, unclipped source profile while keeping the chosen `m_arms`, signed `pitch_angle`, and `Theta0` fixed. Geometry is therefore not re-selected by the downstream source fit.

A row-wise circular scramble preserves each radial row's residual distribution, mask, and coverage while destroying a coherent spiral slope across radius. For each `m`, the best accepted scrambled score supplies a null distribution that corrects the arm-family look-elsewhere bias. `null_z` is an empirical morphology-ranking z-score based on only eight scrambles; it is not Gaussian significance, posterior odds, or a formal uncertainty. Those eight exploratory draws are seed-sensitive. A single deterministic `np.random.default_rng(RNG_SEED)` stream supplies the sequential row shifts, and each physical-width calibration resets that stream so every width uses the same validated shift ensemble.

The same eight-scramble calibration is repeated independently for every physical ridge width. Raw validated scores are never compared across widths because changing the corridor changes the score scale. Width sensitivity therefore compares the independently null-calibrated winner in each run, and the fiducial geometry is selected without hard-coding an arm number or pitch.

In [ ]:
def scramble_logpolar_rows(logpolar_map, seed):
    """Destroy cross-radius spiral coherence but preserve every row and mask."""
    local_rng = np.random.default_rng(seed)
    result = {key: (value.copy() if isinstance(value, np.ndarray) else value)
              for key, value in logpolar_map.items()}
    for row_index in range(len(result["u"])):
        shift = int(local_rng.integers(0, len(result["phi"])))
        for key in ["radial_residual", "local_ridge", "coverage", "valid"]:
            result[key][row_index] = np.roll(result[key][row_index], shift)
    return result


def build_m_null_calibration(logpolar_map, core_width_kpc=RIDGE_CORE_WIDTH_KPC,
                             n_null=RIDGE_N_NULL):
    """Use one fixed sequential RNG stream for exploratory, seed-sensitive nulls."""
    rows = []
    null_rng = np.random.default_rng(RNG_SEED)
    for null_index in range(n_null):
        scrambled = scramble_logpolar_rows(logpolar_map, null_rng)
        _, null_table = held_out_ridge_search(
            scrambled, core_width_kpc=core_width_kpc, allow_no_acceptance=True)
        for m_arms in M_CANDIDATES:
            family = null_table.loc[
                (null_table["m_arms"] == m_arms)
                & (null_table["valid_held_out"] >= RIDGE_MIN_HELD_OUT_SECTORS)
                & np.isfinite(null_table["validated_score"])]
            best_null_score = (float(family["validated_score"].max())
                               if len(family) else 0.0)
            rows.append({"null_index": null_index, "m_arms": int(m_arms),
                         "core_width_kpc": float(core_width_kpc),
                         "best_null_score": best_null_score})
    draws = pd.DataFrame(rows)
    summary = (draws.groupby("m_arms")["best_null_score"]
               .agg(null_mean="mean", null_std="std", null_count="count")
               .reset_index())
    if not (summary["null_count"] == n_null).all():
        raise RuntimeError("Incomplete per-m null calibration")
    summary["null_std_floor"] = np.maximum(summary["null_std"], 1e-6)
    return draws, summary


def apply_m_null_calibration(candidate_table, null_summary):
    calibrated = candidate_table.merge(
        null_summary, on="m_arms", how="left", validate="many_to_one")
    calibrated["null_z"] = ((calibrated["validated_score"] - calibrated["null_mean"])
                            / calibrated["null_std_floor"])
    accepted = calibrated.loc[
        (calibrated["valid_held_out"] >= RIDGE_MIN_HELD_OUT_SECTORS)
        & (calibrated["validated_score"] > 0)
        & np.isfinite(calibrated["null_z"])].copy()
    if accepted.empty:
        raise RuntimeError("No real ridge candidate survived support and null calibration")
    accepted = accepted.sort_values(["null_z", "validated_score"], ascending=False)
    return accepted.iloc[0].to_dict(), calibrated, accepted


_, ridge_candidate_table_raw = held_out_ridge_search(logpolar)
ridge_null_draws, ridge_null_summary = build_m_null_calibration(logpolar)
ridge_geometry, ridge_candidate_table, accepted_real = apply_m_null_calibration(
    ridge_candidate_table_raw, ridge_null_summary)
display(accepted_real.groupby(["m_arms", "winding_sign"], as_index=False)
        .first().sort_values("null_z", ascending=False))
display(ridge_null_summary)
print("RIDGE_NULL_NORMALIZATION_PASS")

ridge_width_rows = []
ridge_width_null_draws = []
for width_kpc in RIDGE_WIDTH_SENSITIVITY_KPC:
    if np.isclose(width_kpc, RIDGE_CORE_WIDTH_KPC):
        width_geometry = ridge_geometry
        width_draws = ridge_null_draws.copy()
    else:
        _, width_table_raw = held_out_ridge_search(
            logpolar, core_width_kpc=float(width_kpc))
        width_draws, width_summary = build_m_null_calibration(
            logpolar, core_width_kpc=float(width_kpc))
        width_geometry, _, _ = apply_m_null_calibration(width_table_raw, width_summary)
    ridge_width_null_draws.append(width_draws)
    ridge_width_rows.append({
        "core_width_kpc": float(width_kpc), "m_arms": int(width_geometry["m_arms"]),
        "pitch_angle": float(width_geometry["pitch_angle"]),
        "Theta0": float(width_geometry["Theta0"]),
        "validated_score": float(width_geometry["validated_score"]),
        "null_z": float(width_geometry["null_z"])})
ridge_width_sensitivity = pd.DataFrame(ridge_width_rows)
ridge_width_null_draws = pd.concat(ridge_width_null_draws, ignore_index=True)
assert len(ridge_width_null_draws) == (
    len(RIDGE_WIDTH_SENSITIVITY_KPC) * RIDGE_N_NULL * len(M_CANDIDATES))
display(ridge_width_sensitivity)
print("RIDGE_WIDTH_SENSITIVITY_COMPLETE")

if ridge_geometry["pitch_angle"] >= 0:
    raise RuntimeError(
        "The sign-neutral local-ridge/null-calibrated search did not recover "
        "the visually supported negative winding; report no acceptable global "
        "logarithmic geometry instead of relabelling a crossing curve as an arm.")
if ridge_geometry["validated_score"] <= 0 or ridge_geometry["null_z"] <= 0:
    raise RuntimeError("Best ridge geometry is not above its per-m null baseline")
print("RIDGE_REAL_FIT_COMPLETE")
print(ridge_geometry)

## 12. Leave-one-sector-out ridge-geometry stability

The old sector bootstrap refitted a Fourier/full-field selector and could let the source-profile optimizer change the geometry, so it has been deleted. This diagnostic instead omits one azimuth sector at a time and reruns the ridge selector. Each omission is ranked with the fiducial per-`m` null calibration, exposing whether arm number, winding, pitch, or pivot-radius phase depends strongly on one part of the disturbed disc. It is a stability diagnostic, not a sampling posterior or formal confidence interval.

In [ ]:
def omit_logpolar_sector(logpolar_map, omitted_sector,
                         n_sectors=RIDGE_N_SECTORS):
    phi = logpolar_map["phi"]
    sector_index = np.floor((phi + np.pi)/(2*np.pi)*n_sectors).astype(int)
    sector_index = np.clip(sector_index, 0, n_sectors - 1)
    result = {key: (value.copy() if isinstance(value, np.ndarray) else value)
              for key, value in logpolar_map.items()}
    result["valid"][:, sector_index == omitted_sector] = False
    result["coverage"][~result["valid"]] = 0.0
    result["radial_residual"][~result["valid"]] = np.nan
    result["local_ridge"][~result["valid"]] = np.nan
    return result


sector_rows = []
for omitted_sector in range(RIDGE_N_SECTORS):
    try:
        _, sector_table = held_out_ridge_search(
            omit_logpolar_sector(logpolar, omitted_sector))
        sector_table = sector_table.merge(
            ridge_null_summary, on="m_arms", how="left", validate="many_to_one")
        sector_table["null_z"] = ((sector_table["validated_score"] - sector_table["null_mean"])
                                  / sector_table["null_std_floor"])
        sector_candidates = sector_table.loc[
            (sector_table["valid_held_out"] >= RIDGE_MIN_HELD_OUT_SECTORS)
            & (sector_table["validated_score"] > 0)
            & np.isfinite(sector_table["null_z"])].sort_values(
                ["null_z", "validated_score"], ascending=False)
        if sector_candidates.empty:
            raise RuntimeError("No null-calibrated sector-omission candidate")
        sector_geometry = sector_candidates.iloc[0].to_dict()
        sector_rows.append({"omitted_sector": omitted_sector, "success": True,
                            "error": "", "m_arms": sector_geometry["m_arms"],
                            "pitch_angle": sector_geometry["pitch_angle"],
                            "Theta0": sector_geometry["Theta0"],
                            "held_out_score": sector_geometry["held_out_score"],
                            "null_z": sector_geometry["null_z"]})
    except Exception as exc:
        sector_rows.append({"omitted_sector": omitted_sector, "success": False,
                            "error": f"{type(exc).__name__}: {exc}"})

ridge_sector_results = pd.DataFrame(sector_rows)
valid_sector_results = ridge_sector_results.loc[ridge_sector_results["success"]].copy()
sector_valid_fraction = len(valid_sector_results) / RIDGE_N_SECTORS
if sector_valid_fraction < 0.8:
    print("MODEL_STABILITY_WARNING: fewer than 80% of sector omissions succeeded")
ridge_m_frequency = (valid_sector_results["m_arms"].value_counts(normalize=True)
                     .sort_index())
ridge_negative_fraction = float((valid_sector_results["pitch_angle"] < 0).mean())
same_m = valid_sector_results[
    (valid_sector_results["m_arms"] == ridge_geometry["m_arms"])
    & (np.sign(valid_sector_results["pitch_angle"])
       == np.sign(ridge_geometry["pitch_angle"]))].copy()
if len(same_m):
    pivot_u = np.log(RIDGE_PIVOT_RADIUS_KPC / R_REF_KPC)
    same_m["pivot_phase"] = (same_m["Theta0"] + same_m["m_arms"]
        / np.tan(np.deg2rad(same_m["pitch_angle"])) * pivot_u)
    ridge_phase_stability = float(np.abs(np.mean(np.exp(1j * same_m["pivot_phase"]))))
else:
    ridge_phase_stability = np.nan
display(ridge_sector_results)
display(ridge_m_frequency.rename("selection_fraction").to_frame())
print(f"Sector omission valid fraction={sector_valid_fraction:.1%}; "
      f"negative-winding fraction={ridge_negative_fraction:.1%}; "
      f"same-m-and-winding pivot-phase concentration={ridge_phase_stability:.3f} "
      f"at R={RIDGE_PIVOT_RADIUS_KPC:.3f} kpc")
if (ridge_m_frequency.max() < 0.8
        or 0.1 < ridge_negative_fraction < 0.9
        or (np.isfinite(ridge_phase_stability) and ridge_phase_stability < 0.8)):
    print("MODEL_STABILITY_WARNING: m, winding, or phase is sector-sensitive; "
          "retain this caution in the final interpretation.")
fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
scatter = ax.scatter(valid_sector_results["omitted_sector"],
    valid_sector_results["pitch_angle"], c=valid_sector_results["m_arms"],
    cmap="viridis", vmin=0.5, vmax=6.5, s=55)
ax.axhline(ridge_geometry["pitch_angle"], color="black", ls="--",
           label="all-sector selection")
ax.set(xlabel="omitted azimuth-sector index", ylabel="selected pitch (deg)",
       title="Leave-one-sector-out ridge-geometry stability")
ax.legend(fontsize=8)
fig.colorbar(scatter, ax=ax, ticks=M_COMPARE, label="selected m")
plt.show()
print("RIDGE_SECTOR_STABILITY_COMPLETE")

## 13. Fixed-geometry KTZ model diagnostics will follow

The plots in this section previously depended on the removed full-field selector and an explicitly retained opposite-winding model. Task 4 will construct the KTZ-compatible source profile at the already frozen ridge geometry, and Task 5 will then add projected skeleton, model, and residual diagnostics derived from that fixed fit. This placeholder prevents stale plots from being mistaken for results of the new selector.

In [ ]:
print("FIXED_GEOMETRY_DIAGNOSTICS_PENDING_TASK_5")

## 14. Fixed-geometry spiral skeleton and arm profile will follow

The former skeleton and arm-profile panel used variables produced by the deleted global optimizer. After Task 4 fits only the source profile at fixed ridge geometry, Task 5 will rebuild these panels from that result and distinguish the data-derived ridge skeleton from source-model maxima. This executable placeholder intentionally presents no geometry or profile before those dependencies exist.

In [ ]:
print("FIXED_GEOMETRY_DIAGNOSTICS_PENDING_TASK_5")

## 15. Fixed-geometry KTZ-compatible parameter table will follow

The previous table reported intervals from the deleted full-model sector bootstrap. Task 4 will provide the source-profile parameters conditional on the frozen ridge geometry, and Task 5 will assemble their diagnostics and stability summaries into the final table. This executable placeholder avoids carrying obsolete bootstrap quantities forward while preserving the notebook's documented cell order for the subsequent implementation.

In [ ]:
print("FIXED_GEOMETRY_PARAMETER_TABLE_PENDING_TASK_5")

## 16. Interpretation limits and the next KTZ step

The fitted parameters describe the best global KTZ-compatible source geometry found in all finite HII SFR bins under the fixed centre, distance, inclination, position angle, and adaptive-bin mask. There is no artificial central hole and no outer-radius quantile cut. One centroid per adaptive bin is used to avoid pseudo-replication, while the bin's member-pixel area ensures that all 681,856 valid HII pixels contribute to the spatial weighting.

`m_arms` is the base global mode, not a catalogue count of visible arm segments. Harmonics of a base `m=1` phase add phase-locked spatial `m=2` and `m=3` power. The candidate table, opposite-winding overlay, skeleton, and residual panels must therefore be interpreted together: a slightly lower global objective does not imply that every asymmetric branch is captured by one pitch angle or that the winding direction is uniquely known. Bootstrap variation across sectors measures sensitivity to which parts of this disturbed disc are sampled.

These parameters can now define an external source-field template for a later oxygen-abundance correlation forward model. They do **not** measure `kappa`, `x0`, `t_star`, enrichment delay, source clustering, arm pattern speed, or arm lifetime. A metallicity analysis should keep the SFR-derived geometry fixed or propagate its bootstrap uncertainty, apply the real oxygen mask and binning to every mock, and compare homogeneous KT18, clustered KT18, and spiral-modulated KTZ models.
